# Lesson 05 - Agentic RAG


## Setup

Dis notebook dey show how Agentic RAG (Retrieval-Augmented Generation) pattern work using Microsoft Agent Framework.

**Prerequisites:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — your Azure AI Search service endpoint
- `AZURE_SEARCH_API_KEY` — your Azure AI Search API key
- Azure OpenAI deployment wey don set via environment variables
- Azure CLI don authenticate (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Wetin be Agentic RAG?

Traditional RAG dey follow one fixed pipeline: dem go first find documents, den generate answer. **Agentic RAG** go further by givin di agent freedom to decide **when** and **how** to find information.

With Agentic RAG, di agent fit:
- **Decide** whether dem need find info before e answer question
- **Choose** which data source or tool e wan ask
- **Evaluate** di info wey e find and do follow-up find if di first one no reach
- **Combine** information from many find steps make e form one clear answer

Dis one make di agent more flexible and correct pass di static find-then-generate pipeline.


## Creating a Search Tool

For Agentic RAG, dem wrap external data sources as **tools** wey the agent fit use anytime e want. Dis one make the agent fit see retrieval as just another action e fit do, no be say e be compulsory step.

Below we go define travel knowledge base and show am as tool wey the agent fit call to find destination information.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Building the RAG Agent

Now we dey create agent wey dem bin tell make e **always find information before e reply**. The agent dey use the `search_travel_knowledge` tool to make sure say im answers base for the knowledge wey dey ground and no to depend on im own training data.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterative Retrieval — The Maker-Checker Pattern

One big advantage of Agentic RAG na **iterative retrieval**. Di agent fit do many rounds of search to check, correct, or add more to di first findings — e be like "maker-checker" workflow:

1. **Maker step**: Di agent go collect initial information and write draft answer.
2. **Checker step**: Di agent go do extra retrievals to confirm details or complete any missing part.

Below, dem ask di agent question wey need am to compare plenty destinations, make e search plenti times.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Summary

For dis lesson, you don learn how to build **Agentic RAG** system using Microsoft Agent Framework:

- **Agentic RAG** dey make agents decide by demself wen to find information, wey make retrieval dey dynamic and no dey fixed.
- **Tools as data sources**: External knowledge bases (like Azure AI Search) dem wrap as tools wey agent fit use.
- **Iterative retrieval**: The maker-checker pattern dey allow agent do many retrieval round — dey search, check, and fine-tune — before e give final answer.

For production, you go change the in-memory `TRAVEL_KNOWLEDGE_BASE` with real Azure AI Search index to fit handle big big travel document retrieval.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document na wen AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator) use translate am. Even tho we dey try make am correct, abeg sabi say automatic translation fit get some mistake or no pure 100%. Di ogbonge original document for e own language na di correct one. If na serious matter, to use human professional translation better. We no go take charge if any wahala or wrong understanding show because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
